# Lab 02 — Agent Memory and Reusable Agent Skills

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-02-agent-memory-and-reusable-agent-skills/lab-02-agent-memory-and-reusable-agent-skills.ipynb)

**Topic:** 1 — Modern Agent Foundations

**Objective:** Implement short-term and long-term agent memory, and package a capability as a reusable skill

Give the agent memory so it stops forgetting between turns, then extract a repeatable procedure into a skill file the agent loads on demand.

Full step-by-step instructions are in the Learner Guide.


In [ ]:
!pip install -q openai python-dotenv


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Short-term memory: the conversation buffer

In Lab 01 the `messages` list was rebuilt on every call, so the agent forgot everything between questions. Moving it outside the function makes it a conversation buffer, so a follow-up such as *"and what about tomorrow?"* now resolves.


In [ ]:
import json
import os
import re
from pathlib import Path

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"
MEMORY_FILE = Path("memory.json")
SKILLS_DIR = Path("skills")
SUMMARY_THRESHOLD = 12  # messages before older turns are compacted


class Conversation:
    """Short-term memory: the running message buffer for this session."""

    def __init__(self, system_prompt: str) -> None:
        self.system_prompt = system_prompt
        self.messages: list[dict] = []

    def add(self, message) -> None:
        self.messages.append(message)

    def payload(self) -> list:
        """System prompt is rebuilt each turn so recalled facts stay fresh."""
        return [{"role": "system", "content": self.system_prompt}] + self.messages


## 2. Long-term memory, exposed to the agent as tools

Short-term memory dies with the process. Long-term memory is a file the agent can read and write through tools. Keep `memory.json` out of Git — it holds user data.


In [ ]:
def _load_memory() -> dict:
    if MEMORY_FILE.exists():
        try:
            return json.loads(MEMORY_FILE.read_text())
        except json.JSONDecodeError:
            return {}
    return {}


def remember_fact(key: str, value: str) -> dict:
    """Store a durable fact about the user under a short key."""
    facts = _load_memory()
    facts[key] = value
    MEMORY_FILE.write_text(json.dumps(facts, indent=2))
    return {"stored": {key: value}, "total_facts": len(facts)}


def recall_facts() -> dict:
    """Return every durable fact stored about the user."""
    return {"facts": _load_memory()}


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "remember_fact",
            "description": (
                "Store a durable fact about the user, such as a preference, name "
                "or recurring detail. Call this whenever the user states something "
                "worth remembering for future sessions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {"type": "string", "description": "Short label, e.g. 'home_city'."},
                    "value": {"type": "string", "description": "The fact to remember."},
                },
                "required": ["key", "value"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recall_facts",
            "description": "Retrieve all durable facts previously stored about the user.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

TOOL_REGISTRY = {"remember_fact": remember_fact, "recall_facts": recall_facts}


## 3. Inject recalled facts into the system prompt

Storing facts is useless if nothing reads them. Build the system prompt from the memory file so recall is automatic rather than something the agent must remember to do.


In [ ]:
BASE_PROMPT = (
    "You are a helpful personal assistant. Use the supplied tools rather than "
    "guessing. When the user states a durable preference or detail, call "
    "remember_fact so you still know it in a future session."
)


def build_system_prompt(skill_text: str = "") -> str:
    facts = _load_memory()
    parts = [BASE_PROMPT]
    if facts:
        rendered = "\n".join(f"- {k}: {v}" for k, v in facts.items())
        parts.append(f"Known facts about the user:\n{rendered}")
    if skill_text:
        parts.append(f"Apply the following skill to this task:\n{skill_text}")
    return "\n\n".join(parts)


## 4. Compact older turns once the buffer grows

An unbounded buffer grows until it blows the context window and your budget. Compact the old turns into one summary message and keep the recent ones verbatim.


In [ ]:
def compact(conversation: Conversation) -> None:
    """Replace older turns with a single summary once the buffer is long."""
    if len(conversation.messages) <= SUMMARY_THRESHOLD:
        return

    keep = conversation.messages[-4:]      # recent turns stay verbatim
    older = conversation.messages[:-4]

    transcript = "\n".join(
        f"{m['role']}: {m.get('content', '')}"
        for m in older
        if isinstance(m, dict) and m.get("content")
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Summarise this conversation in under 100 words. "
                           "Keep decisions, names and numbers.",
            },
            {"role": "user", "content": transcript},
        ],
    )
    summary = response.choices[0].message.content
    conversation.messages = [
        {"role": "assistant", "content": f"[Summary of earlier conversation] {summary}"}
    ] + keep


def run_turn(conversation: Conversation, user_input: str, max_turns: int = 5) -> str:
    conversation.add({"role": "user", "content": user_input})

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=conversation.payload(),
            tools=TOOLS,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            conversation.add({"role": "assistant", "content": message.content})
            compact(conversation)
            return message.content

        conversation.add(message)
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print(f"  Tool call: {name}({arguments})")
            try:
                result = TOOL_REGISTRY[name](**arguments)
            except Exception as exc:
                result = {"error": str(exc)}
            conversation.add(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

    return "Stopped: reached the maximum number of turns."


## 5. Write a skill as a Markdown file

A skill is a named, described procedure the agent follows. The front matter matters: `name` identifies the skill and `description` is what your loader matches on. In the repo this file lives at `skills/trip-briefing.md`; the cell below writes it so the notebook is self-contained.


In [ ]:
SKILLS_DIR.mkdir(exist_ok=True)
(SKILLS_DIR / "trip-briefing.md").write_text("""---
name: trip-briefing
description: Use when the user asks for a travel or trip briefing for a city.
---

# Trip Briefing

Produce a briefing in exactly this structure:

1. **Weather** - current conditions and what to pack.
2. **Getting around** - the main public transport option.
3. **One local tip** - something a first-time visitor would miss.

Rules:
- Keep each section to two sentences or fewer.
- Call recall_facts first; if the user has a stored home_city, note the
  time-zone difference from it.
- If you do not know something, say so rather than inventing it.
""")
print("skill written")


## 6. Load the skill only when the request matches its description

Loading every skill on every request defeats the purpose — the point is to spend context only on what the task needs.


In [ ]:
def load_skills() -> list[dict]:
    """Read each skill file into {name, description, body}."""
    skills = []
    if not SKILLS_DIR.exists():
        return skills
    for path in sorted(SKILLS_DIR.glob("*.md")):
        text = path.read_text()
        match = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
        if not match:
            continue
        front, body = match.groups()
        meta = {}
        for line in front.splitlines():
            if ":" in line:
                key, _, value = line.partition(":")
                meta[key.strip()] = value.strip()
        skills.append(
            {
                "name": meta.get("name", path.stem),
                "description": meta.get("description", ""),
                "body": body.strip(),
            }
        )
    return skills


def select_skill(user_input: str, skills: list[dict]) -> dict | None:
    """Ask the model which skill applies, if any."""
    if not skills:
        return None
    catalogue = "\n".join(f"- {s['name']}: {s['description']}" for s in skills)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Choose the single most applicable skill for the user's request. "
                    f"Available skills:\n{catalogue}\n\n"
                    "Reply with the skill name only, or 'none'."
                ),
            },
            {"role": "user", "content": user_input},
        ],
    )
    choice = response.choices[0].message.content.strip().lower()
    return next((s for s in skills if s["name"].lower() == choice), None)


## 7. Ask a question, storing a fact

The agent should call `remember_fact` and `memory.json` should appear on disk.


In [ ]:
skills = load_skills()
conversation = Conversation(build_system_prompt())


def ask(user_input: str) -> None:
    """One turn: select a skill if one matches, then run the loop."""
    skill = select_skill(user_input, skills)
    if skill:
        print(f"  [skill loaded: {skill['name']}]")
    conversation.system_prompt = build_system_prompt(skill["body"] if skill else "")
    print("Agent:", run_turn(conversation, user_input), "\n")


ask("I live in Singapore.")
print("memory.json:", MEMORY_FILE.read_text() if MEMORY_FILE.exists() else "(none)")


## 8. Confirm recall and skill loading

The first question recalls the stored fact. The second should load `trip-briefing` and follow its three-section structure rather than improvising.


In [ ]:
ask("Where do I live?")
ask("Give me a trip briefing for Tokyo.")
